# Introduction to Kleinian Groups

This notebook provides an interactive introduction to **Kleinian groups** and **hyperbolic geometry**.

Topics covered:
1. Möbius transformations and their types
2. Schottky groups and their limit sets
3. Apollonian gaskets via the Descartes theorem
4. Hausdorff dimension estimation via box counting

**Prerequisites**: numpy, scipy, matplotlib, Pillow

In [ ]:
import sys
sys.path.insert(0, '../src')
sys.path.insert(0, '..')

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from kleinian import (
    Mobius, KleinianGroup,
    compute_limit_set, hausdorff_dim_estimate,
    descartes_step, apollonian_gasket, ford_circles
)
from kleinian.discrete import jorgensen_test

print('Imports successful!')
print(f'numpy version: {np.__version__}')

## Möbius Transformations

A **Möbius transformation** has the form:
$$f(z) = \frac{az + b}{cz + d}, \quad ad - bc \neq 0$$

They form the group $\mathrm{PSL}(2, \mathbb{C})$ under composition (matrix multiplication).

Classification by trace squared $\tau = (a+d)^2$:
- **Parabolic**: $\tau = 4$ (one fixed point, translation-like)
- **Elliptic**: $\tau \in (0,4) \subset \mathbb{R}$ (rotation-like)
- **Hyperbolic**: $\tau \in (4,\infty) \subset \mathbb{R}$ (expansion/contraction)
- **Loxodromic**: $\tau \in \mathbb{C} \setminus \mathbb{R}$ (spiral)

In [ ]:
# Identity transformation
I = Mobius.identity()
print(f'Identity: {I}')
print(f'I(2+3j) = {I(2+3j)}')

# Hyperbolic transformation: z -> 2z
H = Mobius.hyperbolic(2.0)
print(f'\nHyperbolic H (z->2z): {H}')
print(f'H(1) = {H(1.0)}, H(1j) = {H(1j)}')
print(f'Is hyperbolic: {H.is_hyperbolic()}')
print(f'Trace^2 = {H.trace()**2:.4f}')
fps = H.fixed_points()
print(f'Fixed points: {fps}')

# Parabolic transformation: z -> z + 1
P = Mobius.parabolic(1.0)
print(f'\nParabolic P (z->z+1): {P}')
print(f'Is parabolic: {P.is_parabolic()}')
print(f'Trace^2 = {P.trace()**2:.4f}')

# Rotation by pi/3
E = Mobius.rotation(np.pi / 3)
print(f'\nElliptic E (rotation pi/3): {E}')
print(f'Is elliptic: {E.is_elliptic()}')

# Composition and inverse
A = Mobius(2, 1, 1, 1)
A_inv = A.inverse()
composed = A * A_inv
print(f'\nA * A^-1 == I? {composed == I}')

# Isometric circle
center, radius = A.isometric_circle()
print(f'Isometric circle of A: center={center:.4f}, radius={radius:.4f}')

## Schottky Groups

A **Schottky group** is a free group $\Gamma = \langle A, B \rangle \subset \mathrm{PSL}(2,\mathbb{C})$
generated by hyperbolic or loxodromic elements whose isometric circles are pairwise disjoint.

The **Jørgensen inequality** gives a necessary condition for discreteness:
$$|\mathrm{tr}^2(A) - 4| + |\mathrm{tr}([A,B]) - 2| \geq 1$$

The **limit set** of a Schottky group is a Cantor set — a totally disconnected, perfect, nowhere-dense
fractal subset of the Riemann sphere.


In [ ]:
# Build a Schottky group
t = 0.8
A = Mobius(np.cosh(t), np.sinh(t), np.sinh(t), np.cosh(t))
phi = Mobius.rotation(np.pi / 2)
B = A.conjugate_by(phi)

print(f'Generator A: {A}')
print(f'Generator B: {B}')
print(f'Trace(A) = {A.trace():.6f}')
print(f'A is hyperbolic: {A.is_hyperbolic()}')

# Jorgensen test
result = jorgensen_test(A, B)
print(f'\n{result["message"]}')

# Build the group and compute limit set
group = KleinianGroup([A, B], names=['a', 'b'])

print('\nComputing limit set (depth=7)...')
pts = compute_limit_set(group, max_depth=7, tol=1e-6)
print(f'Limit set: {len(pts)} points')

if len(pts) > 0:
    print(f'Re range: [{np.real(pts).min():.4f}, {np.real(pts).max():.4f}]')
    print(f'Im range: [{np.imag(pts).min():.4f}, {np.imag(pts).max():.4f}]')

    fig, ax = plt.subplots(1, 1, figsize=(7, 7))
    ax.scatter(np.real(pts), np.imag(pts), s=0.5, c='cyan', alpha=0.7)
    ax.set_aspect('equal')
    ax.set_facecolor('black')
    ax.set_title(f'Schottky Limit Set ({len(pts)} points, depth=7)')
    ax.set_xlabel('Re(z)')
    ax.set_ylabel('Im(z)')
    plt.tight_layout()
    plt.savefig('schottky_nb.png', dpi=100, bbox_inches='tight', facecolor='black')
    print('Saved schottky_nb.png')
    plt.close()

## Apollonian Gasket

The **Apollonian gasket** is a fractal circle packing generated by repeated application of
the **Descartes Circle Theorem**: given four mutually tangent circles with curvatures
$k_1, k_2, k_3, k_4$:

$$(k_1+k_2+k_3+k_4)^2 = 2(k_1^2+k_2^2+k_3^2+k_4^2)$$

Starting from the Descartes quadruple $(-1, 2, 2, 3)$ (where the outer circle has curvature $-1$),
the gasket fills every interstice with a unique circle.

The centers of the circles are computed via the **complex Descartes theorem**:
$$\bar{z}_4 = \frac{k_1\bar{z}_1 + k_2\bar{z}_2 + k_3\bar{z}_3 \pm 2\sqrt{k_1k_2\bar{z}_1\bar{z}_2 + k_2k_3\bar{z}_2\bar{z}_3 + k_1k_3\bar{z}_1\bar{z}_3}}{k_4}$$

In [ ]:
# Verify Descartes theorem
k1, k2, k3 = -1.0, 2.0, 2.0
k4 = descartes_step(k1, k2, k3)
print(f'Descartes: ({k1}, {k2}, {k3}) -> k4 = {k4:.4f}')

lhs = (k1 + k2 + k3 + k4)**2
rhs = 2 * (k1**2 + k2**2 + k3**2 + k4**2)
print(f'LHS = {lhs:.6f}, RHS = {rhs:.6f}, diff = {abs(lhs-rhs):.2e}')

# Generate gasket
print('\nGenerating Apollonian gasket...')
gasket = apollonian_gasket(k1, k2, k3, k4, max_iter=4)
print(f'Generated {len(gasket)} circles')

# Plot
fig, ax = plt.subplots(1, 1, figsize=(8, 8))

import matplotlib.cm as cm
cmap = cm.get_cmap('plasma')

radii = [r for _, r in gasket if np.isfinite(r) and r > 0]
max_r = max(radii) if radii else 1.0

for center, radius in gasket:
    if not np.isfinite(radius) or radius <= 0 or radius > max_r * 1.5:
        continue
    t = 1.0 - min(radius / max_r, 1.0)
    color = cmap(t)
    circle = patches.Circle(
        (center.real, center.imag), radius,
        fill=False, edgecolor=color, linewidth=0.7
    )
    ax.add_patch(circle)

ax.set_aspect('equal')
ax.autoscale()
ax.set_facecolor('black')
ax.set_title(f'Apollonian Gasket ({len(gasket)} circles, k=(-1,2,2,3))', color='white')
fig.patch.set_facecolor('black')
ax.tick_params(colors='white')
plt.tight_layout()
plt.savefig('apollonian_nb.png', dpi=100, bbox_inches='tight', facecolor='black')
print('Saved apollonian_nb.png')
plt.close()

## Hausdorff Dimension

The **Hausdorff dimension** $\delta = \dim_H(\Lambda)$ of the limit set encodes the
complexity of a Kleinian group. For a convex co-compact group:
$$0 < \delta < 2$$
with $\delta = 1$ for Fuchsian groups acting on the real line.

We estimate $\delta$ via **box counting**:
$$\dim_H \approx \lim_{\varepsilon \to 0} \frac{\log N(\varepsilon)}{\log(1/\varepsilon)}$$
where $N(\varepsilon)$ is the number of $\varepsilon$-boxes that intersect the limit set.

We compute this over a range of $\varepsilon$ values and fit the slope via least squares.

In [ ]:
# Compute limit set for Hausdorff estimation
print('Computing high-resolution limit set...')
pts = compute_limit_set(group, max_depth=8, tol=1e-7)
print(f'Number of points: {len(pts)}')

if len(pts) >= 10:
    hdim = hausdorff_dim_estimate(pts)
    print(f'\nEstimated Hausdorff dimension: {hdim:.4f}')
    print('(Expected: between 1.0 and 1.5 for a Schottky group with t=0.8)')

    # Show the box-counting fit visually
    xs = np.real(pts)
    ys = np.imag(pts)
    x_min, x_max = xs.min(), xs.max()
    y_min, y_max = ys.min(), ys.max()
    span = max(x_max - x_min, y_max - y_min, 1e-10)
    epsilon_range = np.logspace(-1, -3, 20) * span

    log_inv_eps = []
    log_N = []
    for eps in epsilon_range:
        if eps <= 0:
            continue
        ix = np.floor((xs - x_min) / eps).astype(int)
        iy = np.floor((ys - y_min) / eps).astype(int)
        N = len(set(zip(ix, iy)))
        if N > 1:
            log_inv_eps.append(np.log(1.0 / eps))
            log_N.append(np.log(N))

    log_inv_eps = np.array(log_inv_eps)
    log_N = np.array(log_N)

    A_mat = np.vstack([log_inv_eps, np.ones(len(log_inv_eps))]).T
    slope, intercept = np.linalg.lstsq(A_mat, log_N, rcond=None)[0]
    fit_line = slope * log_inv_eps + intercept

    fig, ax = plt.subplots(1, 1, figsize=(7, 5))
    ax.scatter(log_inv_eps, log_N, color='cyan', s=30, zorder=5, label='Box counts')
    ax.plot(log_inv_eps, fit_line, 'r-', linewidth=2,
            label=f'Fit: slope = {slope:.4f}')
    ax.set_xlabel('log(1/epsilon)')
    ax.set_ylabel('log N(epsilon)')
    ax.set_title(f'Box-Counting Dimension Estimate: delta approx {slope:.4f}')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('hausdorff_boxcount.png', dpi=100, bbox_inches='tight')
    print('Saved hausdorff_boxcount.png')
    plt.close()
else:
    print('Not enough points for dimension estimate.')